# TPF Super-Resolución — entrenamiento en Kaggle

**Antes de correr:** panel derecho → Accelerator = **GPU T4**, y **Add Input** con tu dataset.
Ajustá `DATASET` abajo al nombre/slug de tu dataset subido.

In [ ]:
# 1) Copiar el proyecto a /kaggle/working (escribible), buscando src/ donde sea
import os, shutil

# Buscar el directorio que contiene 'src' y 'datasets' (sin importar el anidado del zip)
proj = None
for root, dirs, _ in os.walk('/kaggle/input'):
    if 'src' in dirs and 'datasets' in dirs:
        proj = root
        break
assert proj, 'No encontre una carpeta con src/ y datasets/ dentro de /kaggle/input'
print('Proyecto en:', proj)

for sub in ['src', 'datasets']:
    dst = f'/kaggle/working/{sub}'
    if not os.path.exists(dst):
        shutil.copytree(os.path.join(proj, sub), dst)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
print(os.listdir('/kaggle/working'))

In [ ]:
# 2) Sanity: GPU disponible + gate de evaluacion (bicubic debe dar los valores canonicos)
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
%cd /kaggle/working/src
!python metrics_check.py

In [ ]:
# 3) Entrenar los 4 modelos (2 variantes x 2 factores). ~minutos c/u en T4.
%cd /kaggle/working/src
!python train.py --model fsrcnn     --scale 2 --epochs 300 --patches-per-epoch 12000 --batch 64
!python train.py --model fsrcnn_res --scale 2 --epochs 300 --patches-per-epoch 12000 --batch 64
!python train.py --model fsrcnn     --scale 4 --epochs 300 --patches-per-epoch 12000 --batch 64
!python train.py --model fsrcnn_res --scale 4 --epochs 300 --patches-per-epoch 12000 --batch 64

In [ ]:
# 4) Resumen de resultados + curvas
import json, glob
from IPython.display import Image, display
for h in sorted(glob.glob('/kaggle/working/outputs/*/history.json')):
    d = json.load(open(h)); tag = h.split('/')[-2]
    print(tag, '->', d.get('final'))
for c in sorted(glob.glob('/kaggle/working/outputs/*/curve.png')):
    print(c); display(Image(c))

Los pesos (`best.pth`), métricas (`history.json`) y curvas quedan en `/kaggle/working/outputs/`.
**Descargalos** desde el panel derecho (Output) o con *Save Version* para no perderlos.